In [133]:
import numpy as np
import msicpe.san as san
from plotly import express as px
import scipy.signal as signal
import pandas as pd

PARTIE 1

In [134]:
signal_dict1 = np.load('data_partie_1/violin_C.npz') 
s1 = signal_dict1['s'] # extraction du signal utile
t1 = signal_dict1['t'] # extraction du vecteur temps associé au signal

v1, S1  = san.trans_fourier(s1, t1)

Vr1= np.real(v1)
Sr1 = np.real(S1)
Energie1=np.abs(Sr1)

df = pd.DataFrame({"nu":Vr1,"f0": Energie1,})
fig = px.line(df, x="nu", y=["f0"], title="Fonctions", range_x=[0,5000])
fig.show()

In [135]:
signal_dict2 = np.load('data_partie_1/ecg_signal.npz') 
s2 = signal_dict2['s'] # extraction du signal utile
t2 = signal_dict2['t'] # extraction du vecteur temps associé au signal


v2, S2  = san.trans_fourier(s2 - np.mean(s2), t2)

Vr2= np.real(v2)
Sr2 = np.real(S2)
Energie2 = np.abs(Sr2)


df = pd.DataFrame({"nu":Vr2,"f0": Energie2,})
fig = px.line(df, x="nu", y=["f0"], title="Fonctions", range_x=[0,400])
fig.show()

In [136]:
signal_dict3 = np.load('data_partie_1/piano_G.npz') 
s3 = signal_dict3['s'] # extraction du signal utile
t3 = signal_dict3['t'] # extraction du vecteur temps associé au signal

v3, S3  = san.trans_fourier(s3, t3)

Vr3 = np.real(v3)
Sr3 = np.real(S3)
Energie3 = np.abs(Sr3)

df = pd.DataFrame({"nu":Vr3,"f0": Energie3,})
fig = px.line(df, x="nu", y=["f0"], title="Fonctions", range_x=[0,5000])
fig.show()

In [137]:
def anaSpectr(signal, interval):
    signal_dict = np.load(signal) 
    s = signal_dict['s'] # extraction du signal utile
    t = signal_dict['t'] # extraction du vecteur temps associé au signal
    s = s - np.mean(s) #enleve la composante continue qui fait des pics a 0 Hz

    v, S  = san.trans_fourier(s, t)

    Sr = np.real(S) # amplitudes du domaine fréquentiel réel
    Energie = np.abs(Sr) #spectre
    EnergieN= Energie/np.max(Energie) #spectre normalisé

    # Elog= np.log10(Energie)

    df = pd.DataFrame({"nu":v,"amplitudes": EnergieN,})
    fig = px.line(df, x="nu", y=["amplitudes"], title="Fonctions", range_x=interval)
    # df2= pd.DataFrame({"nu":Vr,"f0": Elog,})
    # fig2 = px.line(df2, x="nu", y=["f0"], title="Fonctions", range_x=interval)
    df3 = pd.DataFrame({"temps":t,"signal": s,})
    fig3 = px.line(df3, x="temps", y=["signal"], title="Fonctions")
    fig3.show()
    fig.show()


anaSpectr('data_partie_1/piano_G.npz', [0,5000])
anaSpectr('data_partie_1/tuning_fork_440Hz.npz', [0,1000])

PARTIE 2 : SIGNAL D'ETUDE 13

In [138]:
def vect_amplitude(nom, harmoniques, showgraph,showharmoniques):
    signal_dict = np.load(f"data_partie_2/{nom}.npz") 
    s = signal_dict['s'] # extraction du signal utile
    t = signal_dict['t'] # extraction du vecteur temps associé au signal
    s= s - np.mean(s) #enleve la composante continue qui fait des pics a 0 Hz
    
    v, S  = san.trans_fourier(s, t)  #trans fourier de notre signal
    Sr = np.real(S) # amplitudes du domaine fréquentiel réel
    # Sn = Sr/np.max(Sr) #domaine fréquentiel réel normalisé
    Energie = np.abs(Sr) #spectre
    EnergieN= Energie/np.max(Energie) #spectre normalisé

    if showgraph==True:
        anaSpectr(f"data_partie_2/{nom}.npz",[0,5000]) #montre le signal et son spectre

    fondamentale = san.detect_fondamentale(EnergieN, v) #fondamentale qui ne correspond pas exactement au graphe affiché (potentiellement plus precis)
    if showharmoniques==True: print("Fondamentale : ",fondamentale) #print la fondamentale

    harmo=[]
    for i in range (0,harmoniques+1):
        harmo.append(float(fondamentale*(i+1))) #calcule tous les harmoniques a partir de la fondamentale
    #print(harmo)

    freq, amplitudes= san.detect_pics(EnergieN, harmo , v) #donne les amplitudes et les fréquences du signal
    amplitudeslog= np.log10(amplitudes)
    if showharmoniques==True: print(amplitudeslog)
    return amplitudeslog

# vect_amplitude("apprentissage/a_5",5, True, True)
# vect_amplitude("apprentissage/a_3",5, True, True)
# vect_amplitude("apprentissage/a_4",5, True, True)

def valmoy(voyelle, harmonique, showgraph, showharmoniques): #pour trouver le vecteur moyen
    liste=[]
    for i in range(1,6):
        liste.append(vect_amplitude(f"apprentissage/{voyelle}_{i}", harmonique, showgraph, showharmoniques))
    print(liste)
    moyenne = np.mean(liste, axis=0) #moyenne les colonnes de chaque sous-liste
    print("Vecteur moyen pour "+voyelle+" :",moyenne)
    return moyenne

# valmoy("a",2, False, False)
# valmoy("e",2, False, False)
# valmoy("i",2, False, False)
# valmoy("o",2, False, False)
# valmoy("u",2, False, False)

def distance(x1,x2):
    d=np.abs(np.subtract(x1,x2))
    print("Distance : ",d)
    return d

# vect_amplitude("a_5",5, False, True)
# valmoy("a",5, False, False)


def classificationCorrecte(voyelleTest,voyellevisee,harmoniques):
    voyelles = ["a","e","i","o","u"]
    listeDistances = []
    x=vect_amplitude(f"test/{voyelleTest}",harmoniques, False, False)

    for i in voyelles:
        d=distance(valmoy(i,harmoniques, False, False), x)
        listeDistances.append(np.sum(d)) #pour comparer, il faut des float et non des listes: on fait donc la somme des amplitudes de chaque liste pour les comparer et trouver leur "distance"

    indice = np.argmin(listeDistances) # position de la plus petite distance
    voyelleTrouvee = voyelles[indice] # va chercher la voyelle qui correspond a la plus petite distance au vecteur moyen
    print(voyelleTrouvee)

    return voyelleTrouvee == voyellevisee.lower() #booléen qui verifie si on a bien classifié


classificationCorrecte("a_7","a", 20)


"""Fonction qui nous a permis de trouver le nombre d'harmoniques :"""
# def testHarmo(harmoniques):
#     signaux = ["a_6","a_7","e_6","e_7","i_6","i_7","o_6","o_7","u_6","u_7"]
#     nbOk = 0
#     for nom in signaux:
#         ok = classificationCorrecte(nom, nom[0], harmoniques)  #retourne true si la voyelle trouvée est la bonne
#         print(nom, "->", "OK" if ok else "ERREUR")
#         nbOk += ok
#     print(f"{nbOk} / {len(signaux)} bien classés")
#     return nbOk == len(signaux)    # True si tout est bon

# liste=[]
# for harmonique in range(1,20):
#     liste.append(testHarmo(harmonique))
# print(liste)

"""Nous a retournée True pour une seule indice de la liste càd l'indice 11 (on commence a 1 donc on prends bien 11 harmoniques) dans notre cas"""

def trouverVoyelle(voyelleTest,harmoniques):
    voyelles = ["a","e","i","o","u"]
    listeDistances = []
    x=vect_amplitude(f"etude/{voyelleTest}",harmoniques, False, False)

    for i in voyelles:
        d=distance(valmoy(i,harmoniques, False, False), x)
        listeDistances.append(np.sum(d)) #pour comparer, il faut des float et non des listes: on fait donc la somme des amplitudes de chaque liste pour les comparer et trouver leur "distance"

    print(listeDistances)

    indice = np.argmin(listeDistances) # position de la plus petite distance
    voyelleTrouvee = voyelles[indice] # va chercher la voyelle qui correspond a la plus petite distance au vecteur moyen
    print(voyelleTrouvee)
    return voyelleTrouvee

trouverVoyelle("signal_13",11) # -> donne le son 'o'


[array([ 0.        , -0.23044543, -0.7780722 , -0.71548265, -0.9058774 ,
       -0.7748314 , -0.42703906, -0.18744913, -0.582068  , -1.2290139 ,
       -0.7343403 , -0.7758563 , -0.90627855, -1.4547774 , -1.8756877 ,
       -1.7948602 , -1.9367677 , -2.1771586 , -2.2461348 , -2.2029712 ,
       -2.0024476 ], dtype=float32), array([ 0.        , -0.25011942, -0.63411444, -0.5083302 , -0.90074795,
       -0.36294025, -0.22766273, -0.5528198 , -0.85129994, -0.9875871 ,
       -0.520279  , -0.9384341 , -1.5056562 , -1.7612354 , -2.1126463 ,
       -2.1790693 , -2.2888536 , -2.2868848 , -2.3273804 , -1.9992927 ,
       -2.2047558 ], dtype=float32), array([-0.450917  ,  0.        , -0.574798  , -0.52244675, -0.11982671,
       -0.05898518, -0.3833732 , -0.5915783 , -0.58371407, -0.99415714,
       -0.6030085 , -1.2812012 , -1.4182494 , -1.9876273 , -2.1344821 ,
       -2.3666909 , -2.43723   , -1.9692911 , -2.2389786 , -2.2585797 ,
       -2.1966991 ], dtype=float32), array([-0.33598742,  0. 

'o'